---
Phase 0-2: 線性代數 — 神經網路的數學語言
---

為什麼線性代數對 ML 重要？

神經網路做的事情，本質上就是：
  「把資料從一個空間，透過一連串的線性變換 + 非線性激活，映射到另一個空間」

這裡的「線性變換」就是矩陣乘法。
理解矩陣的幾何意義，你就能直觀理解神經網路在做什麼。

本檔涵蓋：
  1. 向量的基本概念與運算
  2. 矩陣作為「線性變換」
  3. 特徵值與特徵向量（直覺理解）
  4. 範數（Norm）— 衡量大小的方式
  5. 在神經網路中的實際應用

In [1]:
import numpy as np

# ============================================================================
# 1. 向量 (Vector) — 空間中的一個點或方向
# ============================================================================

In [2]:
print("=" * 60)
print("1. 向量的基本概念")
print("=" * 60)

# 向量可以表示：
#   - 一個資料點的特徵（例如：[身高, 體重, 年齡]）
#   - 一個方向和大小
#   - 神經網路中一層的輸出

v1 = np.array([3, 4])
v2 = np.array([1, 2])

# --- 向量加法（幾何：平行四邊形法則）---
print(f"v1 = {v1}")
print(f"v2 = {v2}")
print(f"v1 + v2 = {v1 + v2}")     # 對應位置相加

# --- 純量乘法（縮放）---
print(f"2 * v1 = {2 * v1}")        # 方向不變，長度加倍
print(f"-1 * v1 = {-1 * v1}")      # 方向反轉

# --- 點積 (Dot Product) ---
# 幾何意義：兩個向量的「相似程度」
# 公式：v1·v2 = |v1| * |v2| * cos(θ)
#   結果 > 0 → 同方向（相似）
#   結果 = 0 → 垂直（無關）
#   結果 < 0 → 反方向（相反）
dot = np.dot(v1, v2)
print(f"\nv1 · v2 = {dot}")       # 3*1 + 4*2 = 11

# 在 ML 中，點積用來衡量兩個向量的相似度
# Attention 機制的核心就是算 Query 和 Key 的點積

1. 向量的基本概念
v1 = [3 4]
v2 = [1 2]
v1 + v2 = [4 6]
2 * v1 = [6 8]
-1 * v1 = [-3 -4]

v1 · v2 = 11


## 2. 矩陣作為「線性變換」

In [3]:
print("\n" + "=" * 60)
print("2. 矩陣 = 線性變換")
print("=" * 60)

# 矩陣乘以向量 = 把向量「變換」到新的位置
# 想像矩陣是一台「機器」，輸入一個向量，輸出另一個向量

# 例：旋轉矩陣（把向量逆時針旋轉 90 度）
theta = np.pi / 2   # 90 度
rotation = np.array([[np.cos(theta), -np.sin(theta)],
                      [np.sin(theta),  np.cos(theta)]])

v = np.array([1, 0])  # 指向右邊的向量
v_rotated = rotation @ v

print(f"原始向量:     {v}")
print(f"旋轉 90° 後: [{v_rotated[0]:.4f}, {v_rotated[1]:.4f}]")  # → [0, 1] 指向上

# 例：縮放矩陣
scale = np.array([[2, 0],
                   [0, 3]])    # x 方向放大 2 倍，y 方向放大 3 倍
v = np.array([1, 1])
print(f"\n原始向量:     {v}")
print(f"縮放後:       {scale @ v}")     # → [2, 3]

# 例：神經網路的一層就是一個線性變換
print("\n--- 神經網路中的線性變換 ---")
# 假設輸入是 3 維特徵，輸出是 2 維
W = np.array([[0.1, 0.2, 0.3],
              [0.4, 0.5, 0.6]])    # shape: (2, 3) → 3 維 → 2 維
x = np.array([1.0, 2.0, 3.0])      # 一個 3 維的輸入

output = W @ x
print(f"輸入 (3 維): {x}")
print(f"權重矩陣 W shape: {W.shape}")
print(f"輸出 (2 維): {output}")
print("→ 矩陣 W 把 3 維空間「壓縮」到了 2 維空間")

# 連續多層 = 連續多次變換
W1 = np.random.randn(4, 3)    # 3 → 4
W2 = np.random.randn(2, 4)    # 4 → 2
x = np.random.randn(3)

h = W1 @ x        # 第一層：3 維 → 4 維
y = W2 @ h        # 第二層：4 維 → 2 維
print(f"\n兩層線性變換: {x.shape} → {h.shape} → {y.shape}")

# 但注意！多層純線性變換 = 一層線性變換
# W2 @ W1 @ x = (W2 @ W1) @ x = W_combined @ x
# 這就是為什麼需要「非線性激活函數」（Phase 2 會學到）


2. 矩陣 = 線性變換
原始向量:     [1 0]
旋轉 90° 後: [0.0000, 1.0000]

原始向量:     [1 1]
縮放後:       [2 3]

--- 神經網路中的線性變換 ---
輸入 (3 維): [1. 2. 3.]
權重矩陣 W shape: (2, 3)
輸出 (2 維): [1.4 3.2]
→ 矩陣 W 把 3 維空間「壓縮」到了 2 維空間

兩層線性變換: (3,) → (4,) → (2,)


## 3. 特徵值與特徵向量（直覺理解）

In [4]:
print("\n" + "=" * 60)
print("3. 特徵值 (Eigenvalue) 與特徵向量 (Eigenvector)")
print("=" * 60)

# 特徵向量：被矩陣變換後，方向不變，只有大小改變的向量
# 特徵值：大小改變的倍數
# A @ v = λ * v  （A 是矩陣，v 是特徵向量，λ 是特徵值）

A = np.array([[3, 1],
              [0, 2]])

eigenvalues, eigenvectors = np.linalg.eig(A)
print(f"矩陣 A =\n{A}")
print(f"特徵值: {eigenvalues}")
print(f"特徵向量:\n{eigenvectors}")

# 驗證：A @ v 應該等於 λ * v
for i in range(len(eigenvalues)):
    v = eigenvectors[:, i]
    lam = eigenvalues[i]
    Av = A @ v
    lv = lam * v
    print(f"\n  特徵值 λ={lam:.1f}, 特徵向量 v={v.round(4)}")
    print(f"  A @ v  = {Av.round(4)}")
    print(f"  λ * v  = {lv.round(4)}")
    print(f"  相等: {np.allclose(Av, lv)}")

# 在 ML 中的應用：
#   - PCA (主成分分析)：找到資料變異最大的方向 = 找共變異數矩陣的特徵向量
#   - 理解模型穩定性：特徵值太大 → 梯度爆炸，特徵值太小 → 梯度消失


3. 特徵值 (Eigenvalue) 與特徵向量 (Eigenvector)
矩陣 A =
[[3 1]
 [0 2]]
特徵值: [3. 2.]
特徵向量:
[[ 1.         -0.70710678]
 [ 0.          0.70710678]]

  特徵值 λ=3.0, 特徵向量 v=[1. 0.]
  A @ v  = [3. 0.]
  λ * v  = [3. 0.]
  相等: True

  特徵值 λ=2.0, 特徵向量 v=[-0.7071  0.7071]
  A @ v  = [-1.4142  1.4142]
  λ * v  = [-1.4142  1.4142]
  相等: True


## 4. 範數 (Norm) — 衡量向量的「大小」

In [5]:
print("\n" + "=" * 60)
print("4. 範數 (Norm)")
print("=" * 60)

v = np.array([3, 4])

# L1 範數 (Manhattan Distance)：各元素絕對值之和
l1 = np.linalg.norm(v, ord=1)
print(f"v = {v}")
print(f"L1 範數: |3| + |4| = {l1}")

# L2 範數 (Euclidean Distance)：各元素平方和的開根號
l2 = np.linalg.norm(v, ord=2)
print(f"L2 範數: sqrt(3² + 4²) = {l2}")

# 在 ML 中的應用：
print(f"""
範數在 ML 中的角色：
  L1 正則化 (Lasso)：讓權重變稀疏（很多變成 0）→ 特徵選擇
  L2 正則化 (Ridge)：讓權重不要太大 → 防止過擬合

  Loss += λ * ||W||₂²   ← L2 正則化加在損失函數上

  這逼迫模型保持權重較小，避免對訓練資料過度擬合
""")

# --- 向量正規化 (Normalization) ---
# 把向量縮放成長度為 1（單位向量），只保留「方向」
v_normalized = v / np.linalg.norm(v)
print(f"正規化後: {v_normalized}")
print(f"正規化後的長度: {np.linalg.norm(v_normalized):.4f}")  # = 1.0

# 在 ML 中：
#   - 計算 Cosine Similarity 前要先正規化
#   - Layer Normalization / Batch Normalization 的基礎概念


4. 範數 (Norm)
v = [3 4]
L1 範數: |3| + |4| = 7.0
L2 範數: sqrt(3² + 4²) = 5.0

範數在 ML 中的角色：
  L1 正則化 (Lasso)：讓權重變稀疏（很多變成 0）→ 特徵選擇
  L2 正則化 (Ridge)：讓權重不要太大 → 防止過擬合

  Loss += λ * ||W||₂²   ← L2 正則化加在損失函數上

  這逼迫模型保持權重較小，避免對訓練資料過度擬合

正規化後: [0.6 0.8]
正規化後的長度: 1.0000


## 5. 實戰：用線性代數模擬一個完整的前向傳播

In [6]:
print("\n" + "=" * 60)
print("5. 實戰：用純 NumPy 模擬神經網路前向傳播")
print("=" * 60)

# 模擬一個 3 層神經網路：
#   輸入：4 個特徵
#   隱藏層 1：8 個神經元
#   隱藏層 2：6 個神經元
#   輸出：3 個類別（分類問題）

np.random.seed(42)

# 模擬一批 5 個樣本，每個有 4 個特徵
X = np.random.randn(5, 4)    # shape: (batch_size=5, features=4)

# 權重和偏差（通常隨機初始化）
W1 = np.random.randn(4, 8) * 0.01   # (4, 8)  4 輸入 → 8 輸出
b1 = np.zeros((1, 8))                # (1, 8)  每個輸出一個 bias
W2 = np.random.randn(8, 6) * 0.01   # (8, 6)  8 輸入 → 6 輸出
b2 = np.zeros((1, 6))
W3 = np.random.randn(6, 3) * 0.01   # (6, 3)  6 輸入 → 3 輸出
b3 = np.zeros((1, 3))

# 激活函數（非線性，Phase 2 會詳細講）
def relu(x):
    return np.maximum(0, x)          # 小於 0 的變成 0，其他不變

def softmax(x):
    exp_x = np.exp(x - x.max(axis=1, keepdims=True))   # 減最大值防溢出
    return exp_x / exp_x.sum(axis=1, keepdims=True)

# 前向傳播
print("各層形狀變化：")
print(f"  輸入 X:  {X.shape}")            # (5, 4)

z1 = X @ W1 + b1                          # 線性變換（broadcasting: b1 廣播到每個樣本）
a1 = relu(z1)                             # 非線性激活
print(f"  第 1 層: {X.shape} @ {W1.shape} + {b1.shape} → {a1.shape}")

z2 = a1 @ W2 + b2
a2 = relu(z2)
print(f"  第 2 層: {a1.shape} @ {W2.shape} + {b2.shape} → {a2.shape}")

z3 = a2 @ W3 + b3
output = softmax(z3)                       # 最後一層用 Softmax 得到機率
print(f"  第 3 層: {a2.shape} @ {W3.shape} + {b3.shape} → {output.shape}")

print(f"\n輸出（每個樣本對 3 個類別的機率）：")
print(output.round(4))
print(f"每列總和（應該都是 1）: {output.sum(axis=1).round(4)}")

# 預測結果
predictions = output.argmax(axis=1)
print(f"預測類別: {predictions}")


5. 實戰：用純 NumPy 模擬神經網路前向傳播
各層形狀變化：
  輸入 X:  (5, 4)
  第 1 層: (5, 4) @ (4, 8) + (1, 8) → (5, 8)
  第 2 層: (5, 8) @ (8, 6) + (1, 6) → (5, 6)
  第 3 層: (5, 6) @ (6, 3) + (1, 3) → (5, 3)

輸出（每個樣本對 3 個類別的機率）：
[[0.3333 0.3333 0.3333]
 [0.3333 0.3333 0.3333]
 [0.3333 0.3333 0.3333]
 [0.3333 0.3333 0.3333]
 [0.3333 0.3333 0.3333]]
每列總和（應該都是 1）: [1. 1. 1. 1. 1.]
預測類別: [1 1 2 2 2]


## 小結

In [7]:
print("\n" + "=" * 60)
print("小結")
print("=" * 60)
print("""
線性代數在 ML 中的角色：

  概念              →  ML 對應
  ────────────────────────────────
  矩陣乘法          →  神經網路每一層的運算 (X @ W + b)
  線性變換          →  把資料從一個空間映射到另一個空間
  轉置              →  反向傳播中計算梯度
  特徵值/特徵向量    →  PCA 降維、理解梯度穩定性
  範數              →  正則化、衡量誤差大小
  向量正規化         →  Normalization 技術

關鍵直覺：
  神經網路 = 一連串的矩陣乘法 + 非線性激活
  訓練模型 = 找到最好的那些矩陣（權重）

下一步：03_calculus.py — 理解「怎麼找到最好的權重」（梯度下降）
""")


小結

線性代數在 ML 中的角色：

  概念              →  ML 對應
  ────────────────────────────────
  矩陣乘法          →  神經網路每一層的運算 (X @ W + b)
  線性變換          →  把資料從一個空間映射到另一個空間
  轉置              →  反向傳播中計算梯度
  特徵值/特徵向量    →  PCA 降維、理解梯度穩定性
  範數              →  正則化、衡量誤差大小
  向量正規化         →  Normalization 技術

關鍵直覺：
  神經網路 = 一連串的矩陣乘法 + 非線性激活
  訓練模型 = 找到最好的那些矩陣（權重）

下一步：03_calculus.py — 理解「怎麼找到最好的權重」（梯度下降）

